# GoEmotions - Machine Learning Pipeline

This notebook is designed to run in **Google Colab**. It automatically clones the repository, installs the package, and runs the full pipeline.

**Architecture:** Both traditional features (TF-IDF, BoW) and modern features (Word2Vec, BERT) produce the same dense numpy arrays, allowing the exact same classifiers to be used on both.

In [ ]:
# ── 1. Setup ──────────────────────────────────────────
import os, sys

if 'google.colab' in sys.modules:
    REPO_URL = 'https://github.com/TCL03-HCMUT/ML-Assignment.git'
    if not os.path.exists('/content/goemotions-ml-assignment'):
        !git clone {REPO_URL} /content/goemotions-ml-assignment
    %cd /content/goemotions-ml-assignment
    !pip install -e .
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
# ── 2. Imports ─────────────────────────────────────────
from modules.config import (
    PreprocessingConfig,
    TraditionalFeatureConfig,
    ModernFeatureConfig,
    ClassifierConfig,
    DLPipelineConfig,
    EvaluationConfig,
    LABEL_NAMES,
)
from modules.data_loader import load_goemotions_data, get_eda_statistics, plot_class_distribution, plot_text_length_distribution, plot_label_cooccurrence
from modules.preprocessing import preprocess_dataset
from modules.feature_io import save_features, load_features
from modules.traditional_pipeline import TraditionalFeatureExtractor
from modules.modern_features import ModernFeatureExtractor
from modules.classifier import Classifier
from modules.dl_pipeline import EndToEndDLPipeline
from modules.evaluation import compute_metrics, compare_pipelines, plot_per_class_f1

print('All modules imported successfully!')

## 3. Data Loading & EDA

In [ ]:
df_train, df_val, df_test = load_goemotions_data()
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')

stats = get_eda_statistics(df_train)
print(stats)

plot_class_distribution(df_train)
plot_text_length_distribution(df_train)
plot_label_cooccurrence(df_train)

## 4. Preprocessing

In [ ]:
preproc_cfg = PreprocessingConfig(
    lowercase=True,
    remove_punctuation=False,
    remove_stopwords=False,
    stemming=False,
    lemmatization=False,
)

df_train = preprocess_dataset(df_train, preproc_cfg)
df_val   = preprocess_dataset(df_val, preproc_cfg)
df_test  = preprocess_dataset(df_test, preproc_cfg)

# Extract labels as numpy arrays
y_train = df_train[LABEL_NAMES].values
y_val   = df_val[LABEL_NAMES].values
y_test  = df_test[LABEL_NAMES].values

## 5. Feature Extraction

Both extractors produce dense numpy arrays of shape `(N, D)` and save them to `features/`.

**Key difference:**
- **Traditional** (TF-IDF, BoW): The vectorizer must be `fit` on training data only (`is_train=True`), then `transform` on val/test (`is_train=False`). This is the standard sklearn pattern.
- **Modern** (Word2Vec, BERT): No fitting step — models are pre-trained, so `extract()` works identically on all splits. No `is_train` parameter needed.

In [ ]:
# ── 5a. Traditional Features (TF-IDF) ──────────────────
# The vectorizer is fit on training data ONLY (is_train=True).
# Val and test reuse the SAME fitted vocabulary (is_train=False).
trad_cfg = TraditionalFeatureConfig(
    feature_type='tfidf',
    ngram_range=(1, 2),
    max_features=10_000,
)
trad_extractor = TraditionalFeatureExtractor(trad_cfg)

X_tfidf_train = trad_extractor.extract_and_save(df_train['processed_text'], 'tfidf_train', is_train=True)   # fit + transform
X_tfidf_val   = trad_extractor.extract_and_save(df_val['processed_text'],   'tfidf_val',   is_train=False)  # transform only
X_tfidf_test  = trad_extractor.extract_and_save(df_test['processed_text'],  'tfidf_test',  is_train=False)  # transform only

In [ ]:
# ── 5b. Modern Features (e.g. BERT frozen) ─────────────
# No fitting step — the pre-trained model is used as-is on all splits.
# extract() works identically regardless of the split.
modern_cfg = ModernFeatureConfig(
    embedding_type='bert',
    model_name='bert-base-uncased',
    pooling_strategy='cls',
)
modern_extractor = ModernFeatureExtractor(modern_cfg)

X_bert_train = modern_extractor.extract_and_save(df_train['text'].tolist(), 'bert_cls_train')
X_bert_val   = modern_extractor.extract_and_save(df_val['text'].tolist(),   'bert_cls_val')
X_bert_test  = modern_extractor.extract_and_save(df_test['text'].tolist(),  'bert_cls_test')

## 6. Classification (Shared)

The **same** `Classifier` is used on both TF-IDF and BERT features.
Both feature arrays are just `(N, D)` numpy matrices — the classifier doesn't care where they came from.

In [ ]:
clf_cfg = ClassifierConfig(
    classifier_type='logistic_regression',
    tuning_method=None,  # Set to 'grid' or 'random' to enable tuning
)

# ── On Traditional Features ────────────────────────────
clf_trad = Classifier(clf_cfg)
y_pred_trad = clf_trad.run(X_tfidf_train, y_train, X_tfidf_test)

# ── On Modern Features ─────────────────────────────────
clf_modern = Classifier(clf_cfg)
y_pred_modern = clf_modern.run(X_bert_train, y_train, X_bert_test)

## 7. End-to-End Deep Learning (Bonus)

In [ ]:
dl_cfg = DLPipelineConfig(
    model_name='distilbert-base-uncased',
    epochs=3,
    learning_rate=2e-5,
    batch_size=16,
)
dl_pipeline = EndToEndDLPipeline(dl_cfg)
y_pred_dl = dl_pipeline.run(
    df_train['text'].tolist(), y_train,
    df_val['text'].tolist(), y_val,
    df_test['text'].tolist(), y_test,
)

## 8. Evaluation & Comparison

In [ ]:
eval_cfg = EvaluationConfig(show_classification_report=True)

metrics_trad   = compute_metrics(y_test, y_pred_trad,   eval_cfg)
metrics_modern = compute_metrics(y_test, y_pred_modern, eval_cfg)
metrics_dl     = compute_metrics(y_test, y_pred_dl,     eval_cfg)

compare_pipelines({
    'TF-IDF + LogReg':       metrics_trad,
    'BERT Frozen + LogReg':  metrics_modern,
    'DistilBERT E2E':        metrics_dl,
})

plot_per_class_f1(y_test, y_pred_dl, eval_cfg)